# 프로젝트 1 - Weekend 1: API와 체인 기반 FAQ 시스템 (Easy)

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 |
| **소요 시간** | 5시간 (10사이클 x 30분) |
| **핵심 기술** | Python, OpenAI API, LangChain LCEL, Gradio |

## 환경 설정

In [ ]:
# !pip install -q openai langchain-openai python-dotenv gradio

In [ ]:
# colab 사용 시 아래 주석 해제
# import os
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
# from openai import OpenAI
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# client = OpenAI()
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# 환경 설정
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ 환경 설정 완료")

## 📦 실습용 샘플 데이터

In [ ]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

---
## 사이클 1: 첫 API 호출

OpenAI API로 주택청약 관련 질문을 보내고 답변을 받아보세요. `system` 역할에 "주택청약 전문 상담원"을 설정하세요.

In [ ]:
# 사이클 1


---
## 사이클 2: FAQ 데이터 탐색

`SAMPLE_FAQ_DATA`에서 카테고리별 FAQ 개수를 세고, `difficulty`가 `"easy"`인 항목만 필터링해서 출력하세요.

In [ ]:
# 사이클 2


---
## 사이클 3: FAQ 검색 함수

질문 문자열을 받아 키워드 매칭으로 관련 FAQ를 찾는 `search_faq(query, faq_data, top_k=3)` 함수를 만들고, `SAMPLE_TEST_QUERIES`로 테스트하세요.

In [ ]:
# 사이클 3


---
## 사이클 4: 검색 결과 + LLM 답변 생성

검색된 FAQ를 system prompt에 넣어 답변을 생성하는 `ask_faq(question, faq_data, client)` 함수를 만드세요. 답변과 함께 참고한 FAQ 목록도 반환하세요.

In [ ]:
# 사이클 4


---
## 사이클 5: PromptTemplate

`ChatPromptTemplate`으로 `{context}`와 `{question}` 변수를 사용하는 FAQ 답변용 프롬프트를 만들고, 카테고리 분류용 프롬프트도 하나 더 만들어서 각각 테스트하세요.

In [ ]:
# 사이클 5
from langchain_core.prompts import ChatPromptTemplate


---
## 사이클 6: LCEL 체인

`prompt | llm | StrOutputParser()` 패턴으로 FAQ 답변 체인(`faq_chain`)을 만들고, 질문 2개로 테스트하세요. `.stream()`으로 스트리밍 출력도 해보세요.

In [ ]:
# 사이클 6
from langchain_core.output_parsers import StrOutputParser


---
## 사이클 7: 검색 

질문을 넣으면 자동으로 FAQ 검색 → 답변 생성하는 `rag_chain`을 만드세요. `SAMPLE_TEST_QUERIES` 5개로 테스트하세요.

In [ ]:
# 사이클 7


---
## 사이클 8: 에러 처리

빈 입력, 500자 초과, 숫자만 입력 등을 검증하고 `try/except`로 API 오류를 처리하는 `safe_ask(question, rag_chain)` 함수를 만드세요. 정상/에러 케이스 6가지 이상 테스트하세요.

In [ ]:
# 사이클 8
import time


---
## 사이클 9: Gradio 채팅 UI

`gr.ChatInterface`로 지금까지 만든 RAG 체인을 웹 채팅 UI로 만드세요. 제목, 설명, 예시 질문 5개를 설정하세요.

In [ ]:
# 사이클 9
import gradio as gr


---
## 사이클 10: 최종 통합 테스트

전체 파이프라인(입력 검증 → 검색 → 답변 생성)을 하나의 함수로 정리하고, 10개 질문으로 테스트하세요. 각 질문의 응답 시간, 참고 FAQ 수를 포함한 결과표를 출력하세요. Gradio UI도 최종 버전으로 만드세요.

In [ ]:
# 사이클 10
import gradio as gr
import time


---
## Weekend 1 완료! 🎉

다음 주: 벡터 임베딩 + RAG 파이프라인